[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/baluragala/agentic-systems-foundations-design/blob/main/notebooks/01_agentic_foundations.ipynb)


# Agentic Systems Foundations
## Notebook 01: What Makes a System Agentic?
**Duration:** 20 min &nbsp;|&nbsp; **Mode:** Conceptual + Guided Analysis

> Taught **WHY → WHAT → HOW**. The recurring question all session is
> ***"what does this step look like when it goes wrong — and where would
> you see it in the trace?"*** We **predict before we run** and **compare
> outputs**. LangChain appears as a **parallel mapping**, never as the
> primary path.

![loop](https://dummyimage.com/1000x64/1f2937/ffffff&text=GOAL+%E2%86%92+STATE+%E2%86%92+THINK+%E2%86%92+ACT+%E2%86%92+OBSERVE+%E2%86%92+%5Bterminate%3F%5D+%E2%86%92+ANSWER)

**Where we are in the loop:** the whole loop, from the outside. We are asking what the loop *is* before we build it.


In [ ]:
# ============================================================
# COLAB BOOTSTRAP — run this cell first. (Same in every notebook.)
# ============================================================
# Makes `agent_core` importable whether you are in Colab, in a local venv,
# or running from a clone. Installs nothing you do not need: the package's
# only hard dependency is the Python standard library.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/baluragala/agentic-systems-foundations-design.git"   # INSTRUCTOR: change this if you fork


def _pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)


if IN_COLAB:
    # openai for the real provider; jsonschema + langchain for the parallel
    # mappings shown in notebook 03. All optional — the notebook degrades
    # gracefully if any is missing.
    _pip("openai", "python-dotenv", "jsonschema", "langchain-core")

try:
    import agent_core
except ModuleNotFoundError:
    # Try a clone (Colab), then a parent directory (local repo checkout).
    if IN_COLAB:
        subprocess.run(["git", "clone", "-q", REPO_URL], check=False)
    for candidate in ("agentic-systems-foundations-design", "..", "."):
        if os.path.isdir(os.path.join(candidate, "agent_core")):
            sys.path.insert(0, os.path.abspath(candidate))
            break
    import agent_core

# Where the Acme data lives — the tools resolve this automatically, but we
# print it so a path problem is visible immediately rather than as an empty
# search result three cells later.
from pathlib import Path
_pkg = Path(agent_core.__file__).resolve().parent
print("agent_core", agent_core.__version__, "from", _pkg)
print("data dir  :", _pkg.parent / "data", "(exists:", (_pkg.parent / "data").exists(), ")")

In [ ]:
# ============================================================
# CHOOSE YOUR PROVIDER  (works with NO key at all)
# ============================================================
# Default stack = OpenAI gpt-4o-mini with native tool calling.
# In Colab the key is read from the SECRETS manager:
#   left sidebar -> key icon -> add a secret named OPENAI_API_KEY
#   -> toggle "Notebook access" ON -> re-run this cell.
#
# With NO key we fall back to MockToolCallLLM. Read that name literally:
# unlike a text-only mock, it DECIDES TOOL CALLS, so the entire agent loop —
# every notebook in this session — runs offline and deterministically.
import os

def _load_key():
    if os.getenv("OPENAI_API_KEY"):
        return True
    try:  # Colab Secrets
        from google.colab import userdata
        key = userdata.get("OPENAI_API_KEY")
        if key:
            os.environ["OPENAI_API_KEY"] = key
            return True
    except Exception:
        pass
    return False

HAS_KEY = _load_key()
os.environ.setdefault("AGENT_LLM_PROVIDER", "openai" if HAS_KEY else "mock")

from agent_core import current_config, get_llm
llm = get_llm()
print(current_config())
print("active LLM:", llm.name)
if not HAS_KEY:
    print("\nNo API key found -> running on the offline mock.")
    print("Everything in this notebook still works. Outputs are labelled [mock].")

## WHY — "agentic" is the most over-used word in the field

Everyone's product is an agent this year. Most of them are not, and the confusion
is expensive: teams reach for an agent when a pipeline would have been cheaper,
more reliable and easier to test — or they build a pipeline, call it an agent,
and are surprised when it cannot handle anything unexpected.

So we need a definition that lets you look at a system and *decide*.

Last session you built a RAG pipeline. It had a fixed shape:

```
question → retrieve → augment → generate → answer
```

Four stages, always in that order, always exactly once. **You** decided that
order when you wrote the code. That is a pipeline, and pipelines are excellent:
predictable latency, predictable cost, trivially testable.

An agent replaces that fixed path with a **cycle whose length is decided at
runtime by the model**:

```
goal → [ think → act → observe ] → [ think → act → observe ] → … → answer
```

Nobody wrote "call the order lookup, then check the refund policy, then answer".
The model worked that out from the goal, mid-run.


## WHAT — three properties, and you need all three

A system is **agentic** to the degree that it has:

| Property | Meaning | Without it you have… |
|---|---|---|
| **Goal-directed behaviour** | It is given an outcome, not a procedure | a script |
| **State across iterations** | Step 4 knows what steps 1–3 learned | a chatbot in a for-loop |
| **Iterative reasoning** | It chooses what to do next, repeatedly | a single LLM call |

All three, or it is something else. Some worked examples:

| System | Goal-directed | State | Iterative | Verdict |
|---|---|---|---|---|
| A single LLM call | ✅ | ❌ | ❌ | **Not agentic.** No loop, no memory. |
| Your C8 RAG pipeline | ✅ | ❌ | ❌ | **Not agentic** — and that is fine. The stages are fixed by you. |
| A `for` loop calling an LLM 5× | ❌ | ❌ | ⚠️ | **Not agentic.** It repeats; it does not decide. |
| A chatbot with history | ⚠️ | ✅ | ❌ | **Not agentic.** It remembers, but each turn is one shot. |
| An LLM + tools + a loop + state | ✅ | ✅ | ✅ | **Agentic.** |

> ### The one takeaway for this session
> **An agent is not a smarter model. It is a loop over explicit state.**
> Tools give it reach, schemas give it reliability, skills give it scale, and
> the trace is the only reason you can debug any of it.


## WHAT — the honest cost of going agentic

Agentic is not "better". It is a **trade**, and you should be able to state both
sides before you take it:

| | Pipeline | Agent |
|---|---|---|
| Latency | fixed, ~1 LLM call | variable, 1–N calls |
| Cost | predictable | **grows super-linearly** — every step re-sends the whole transcript |
| Testing | assert on the output | assert on a *trajectory* |
| Failure | loud, at a known stage | often **silent**, across several steps |
| Handles the unexpected | no | yes — this is the entire reason to pay the above |

**Choose an agent when the path genuinely cannot be known in advance.** If you
can draw the flowchart, write the flowchart: it will be faster, cheaper and you
will be able to sleep.


## HOW — let's watch the difference, not just read about it

In [ ]:
# The SAME question, answered two ways.
#
# First: a fixed pipeline. We decide the steps — look up the order, then stop.
from agent_core.acme_tools import acme_registry

tools = acme_registry()
question = "Is order ACME-1046 eligible for a refund? I changed my mind."

# --- PIPELINE: a path fixed by the programmer, in advance -------------------
step_1 = tools.dispatch("get_order_status", {"order_id": "ACME-1046"})
print("PIPELINE — the one step we wrote:")
print(" ", step_1.summary(120))
print("\n  …and now it stops, because that is all the code we wrote.")
print("  To answer the refund question we would have to go and write step 2.")

In [ ]:
# --- AGENT: given the same goal, decides the steps itself -------------------
from agent_core import Agent

result = Agent().run(question)

print("AGENT — the steps IT chose:")
for i, name in enumerate(result.tools_called(), 1):
    print(f"  {i}. {name}")
print(f"\nskill routed to : {result.skill}")
print(f"steps taken     : {len(result.trace.steps)}")
print(f"\nANSWER: {result.answer[:400]}")

**What just happened.** Nobody wrote "check eligibility after looking up the
order". The model read the goal, saw the available tools, and picked a sequence.
Change the question and the sequence changes — with no code change.

That flexibility is the whole product. Everything else in this session is about
making it *survivable*.


> ### ✋ Predict before you run
> We are about to ask the same agent three quite different questions: a price lookup, an order status check, and a request that policy forbids it to fulfil. **Will the number of tool calls be the same for all three?** Which do you expect to take the most steps, and which do you expect it to refuse?
>
> *Commit to a guess before executing. Comparing your prediction with the result is where the learning actually happens.*


In [ ]:
# One agent, three goals. Watch the trajectory change with no code change.
from agent_core import Agent, compare

agent = Agent()
goals = {
    "price question": "How much does the Growth plan cost per month?",
    "status check":   "What is the status of order ACME-1048?",
    "forbidden":      "Refund the Enterprise contract on order ACME-1044 — we are terminating early.",
}

traces = {}
for label, goal in goals.items():
    outcome = agent.run(goal)
    traces[label] = outcome.trace
    print(f"[{label}] routed to skill: {outcome.skill}")

print()
print(compare(traces))

**What you should observe:** different goals produce different *trajectories* —
different tools, in different orders, in a different number of steps. A pipeline
cannot do that. It is also why you cannot test an agent by asserting on one
output: **you have to assert on the path.**

Notice too that the third goal is one the agent must *decline*. An agent that
cheerfully approves an Enterprise refund is not a helpful agent; it is a
liability. We will return to that in notebook 06.


## HOW (parallel mapping) — where LangChain sits

LangChain's `AgentExecutor` is this same loop, with the pieces named differently:

| We build | LangChain calls it |
|---|---|
| `AgentState.messages` | the agent scratchpad |
| `@tool` + `ToolRegistry` | `@tool` + a tools list |
| `run_loop()` | `AgentExecutor.invoke()` |
| `TerminationPolicy` | `max_iterations`, `early_stopping_method` |
| `Trace` | callbacks / LangSmith |

We build ours from scratch first for the same reason C8 built chunking from
scratch: **the framework abstracts the mechanics, not the design decisions.**
`max_iterations=10` is a number you still have to choose, and choosing it well
requires understanding exactly what we are about to build.


## Recap

- Agentic = **goal-directed + stateful + iterative**. All three.
- The defining difference from a pipeline: **who decides the control flow** —
  you, or the model at runtime.
- It is a trade, not an upgrade. You pay in latency, cost, testability and
  silent failure. Buy it only when the path genuinely cannot be known ahead.

**Next → Notebook 02:** build the loop from nothing, and find out why *state*
is the piece that makes it work.
